# Chapter 20 — Campaigns, Release Gates, and Provenance

This notebook accompanies **Chapter 20** of *How to Test and Validate Agentic AI Systems*.

**Learning objectives:**
- Orchestrate multi-run campaigns
- Configure risk-tiered release gates
- Trace provenance from document to verdict
- Manage regression suites and version test artifacts

**Modules:** `knowlytix.harness.testing.audit`, `knowlytix.harness.testing.runs`, `knowlytix.harness.testing.replay`,
`knowlytix.harness.testing.provenance`, `knowlytix.harness.testing.snapshots`, `knowlytix.harness.testing.profiling`,
`knowlytix.harness.testing.cost_tracker`, `knowlytix.harness.testing.versioning`

In [ ]:
# === Preamble ===
import sys
import os
import warnings
import io
import contextlib
import shutil
warnings.filterwarnings("ignore")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Clean lab directories
for d in ["monograph/lab_runs", "monograph/lab_regression",
          "monograph/lab_snapshots", "monograph/lab_artifacts"]:
    if os.path.exists(d):
        shutil.rmtree(d)

In [ ]:
from importlib.resources import files
# Load the store and run the benchmark (suppress verbose output)
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    bench = DOEGMSBenchmark(config)
    store = bench.ingest()
    from knowlytix.harness.fixtures import create_full_evaluator
    evaluator = create_full_evaluator()
    result = bench.run(evaluator=evaluator)

print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples")
print(f"Benchmark: {result.total_questions} questions, "
      f"{result.total_correct} correct, "
      f"{result.accuracy:.1f}% accuracy")

## Run Management: State Machine Lifecycle

The `RunManager` tracks benchmark runs through a state machine:
CREATED -> PREPARING -> EXECUTING -> EVALUATING -> COMPLETED (or FAILED/ABORTED).

In [ ]:
from knowlytix.harness.testing.runs import RunManager, RunState

run_mgr = RunManager(runs_dir="monograph/lab_runs")

# Create and transition Run 1
r1 = run_mgr.create_run(config)
print(f"=== Run 1: {r1.run_id} ===")
print(f"  state: {r1.state.value}")

r1 = run_mgr.transition(r1.run_id, RunState.PREPARING)
r1 = run_mgr.transition(r1.run_id, RunState.EXECUTING)
r1 = run_mgr.transition(r1.run_id, RunState.EVALUATING)
r1.accuracy = result.accuracy
r1.n_questions = result.total_questions
r1.n_correct = result.total_correct
r1 = run_mgr.transition(r1.run_id, RunState.COMPLETED)

print(f"  final state: {r1.state.value}")
print(f"  accuracy: {r1.accuracy:.1f}%")
print("\n  State history:")
for ts, from_s, to_s in r1.state_history:
    print(f"    {ts[:19]} : {from_s} -> {to_s}")

# Create Run 2 (simulated improvement)
r2 = run_mgr.create_run(config)
r2 = run_mgr.transition(r2.run_id, RunState.PREPARING)
r2 = run_mgr.transition(r2.run_id, RunState.EXECUTING)
r2 = run_mgr.transition(r2.run_id, RunState.EVALUATING)
r2.accuracy = result.accuracy + 5.0
r2.n_questions = result.total_questions
r2.n_correct = result.total_correct + 32
r2 = run_mgr.transition(r2.run_id, RunState.COMPLETED)
print(f"\n=== Run 2: {r2.run_id} ===")
print(f"  accuracy: {r2.accuracy:.1f}%")

runs = run_mgr.list_runs()
print(f"\n  Total runs: {len(runs)}")

## Release Gates

Release gates enforce quality thresholds before an agent can be
deployed. GMSH supports both simple gates and risk-tiered gates.

In [ ]:
from knowlytix.harness.testing.audit import (
    AuditReporter, RiskTierProfile,
)

# Define risk tier profiles
profiles = [
    RiskTierProfile.advisory(),
    RiskTierProfile.recommendation(),
    RiskTierProfile.action_taking(),
]

print("=== Risk Tier Profiles ===")
print(f"{'Tier':<20s} {'Accuracy':>10s} {'Geo Conf':>10s} {'Tension':>10s} {'Failure Budget'}")
print("-" * 75)
for p in profiles:
    budget_str = str(p.failure_budget) if p.failure_budget else "{}"
    print(f"  {p.tier_name:<18s} {p.accuracy_threshold:>9.0%} "
          f"{p.geo_confidence_threshold:>10.2f} "
          f"{p.max_tension:>10.1f} {budget_str}")

In [ ]:
# Simple release gate
reporter = AuditReporter(result, bench.typed_verdicts)

gate = reporter.generate_release_gate(
    accuracy_threshold=0.80,
    geo_confidence_threshold=0.50,
)

print("=== Simple Release Gate ===")
print(f"  Passed:   {gate.passed}")
print(f"  Accuracy: {gate.actual_accuracy:.1f}% (>= {gate.accuracy_threshold:.0%})")
print(f"  Geo conf: {gate.actual_geo_confidence:.4f} (>= {gate.geo_confidence_threshold})")
print("  Blocking reasons:")
for bf in gate.blocking_failures:
    print(f"    - {bf}")

In [ ]:
# Tiered release gates
tiered_gates = reporter.generate_tiered_release_gate(profiles)

print("=== Tiered Release Gates ===")
for tg in tiered_gates:
    status = "PASS" if tg.passed else "FAIL"
    print(f"\n  [{status}] {tg.tier.tier_name}")
    if tg.checks:
        for ck, cd in tg.checks.items():
            m = "+" if cd.get('passed') else "-"
            print(f"    [{m}] {ck}: {cd.get('actual','')} "
                  f"(>= {cd.get('threshold','')})")
    for reason in tg.blocking_reasons:
        print(f"    BLOCK: {reason}")

## Audit Packets and Failure Reports

The `AuditReporter` generates structured audit trails for every
question and aggregates failure patterns.

In [ ]:
# Audit packets
packets = reporter.generate_audit_packets()

print(f"=== Audit Packets: {len(packets)} ===")
print(f"{'QID':<35s} {'Correct':>8s} {'Geo':>8s} {'Claims':>7s}")
print("-" * 62)
for p in packets[:8]:
    print(f"  {p.doe_qid[:33]:<33s} {str(p.correct):>8s} "
          f"{p.geodesic_confidence:>8.4f} {p.n_claims:>7d}")

# Failure report
fail_report = reporter.generate_failure_report()
print("\n=== Failure Report ===")
print(f"  Total failures: {fail_report.get('total_failures', 0)}")
by_code = fail_report.get('by_code', {})
if by_code:
    for code, cnt in list(by_code.items())[:5]:
        print(f"    {code}: {cnt}")
else:
    print("  No typed failures (typed verdicts not enabled)")

by_cat = fail_report.get('by_category', {})
if by_cat:
    print("\n  By category:")
    for cat, cnt in list(by_cat.items())[:5]:
        print(f"    {cat}: {cnt}")

## Provenance: Document to Verdict

The `ProvenanceRegistry` tracks the lineage from source document
through sections, triples and verdicts.

In [ ]:
from importlib.resources import files
from knowlytix.harness.testing.provenance import (
    ProvenanceRegistry,
)

prov = ProvenanceRegistry()
md_text = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()
doc_rec = prov.register_document(
    str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")), md_text
)

print("=== ProvenanceRegistry ===")
print(f"  doc_id:   {doc_rec.doc_id[:20]}...")
print(f"  source:   {doc_rec.source_path.split('/')[-1]}")
print(f"  version:  {doc_rec.version}")
print(f"  sections: {len(doc_rec.sections)}")
print("\n  Section spans:")
for s in doc_rec.sections[:8]:
    print(f"    [{s.level}] {s.tag[:45]:<45s} ({s.length:>5d} chars)")

In [ ]:
# Link triples to source sections
provenance_links = [
    ("MDL-001", "has_risk_rating", "High", "model_inventory_summary"),
    ("MDL-001", "has_model_name", "CreditScore-XGB", "model_inventory_summary"),
    ("MDL-002", "has_algorithm", "LSTM (layers=3)", "table_22__model_architecture_details"),
    ("MDL-001", "has_test_auc", "0.8412", "table_31__accuracy_diagnostics"),
]

for h, r, t, sec in provenance_links:
    prov.link_triple_to_source(h, r, t, doc_rec.doc_id, sec)

print("=== Provenance Links ===")
for h, r, t, _ in provenance_links:
    result_prov = prov.get_provenance(h, r, t)
    if result_prov:
        doc_id, section = result_prov
        print(f"  ({h}, {r}, {t})")
        print(f"    -> doc: {doc_id[:16]}..., section: {section}")
    else:
        print(f"  ({h}, {r}, {t}) -> NOT FOUND")

## Lab 20.1 — Complete Governance Pipeline

Combine run management, release gates, regression promotion,
snapshots and artifact versioning into a single pipeline.

In [ ]:
# Lab 20.1a -- SnapshotManager: capture store state
from knowlytix.harness.testing.snapshots import SnapshotManager

snap_mgr = SnapshotManager(snapshot_dir="monograph/lab_snapshots")
snap = snap_mgr.take_snapshot(
    store, run_id=r1.run_id, trigger="pre_run"
)

print("=== Snapshot ===")
print(f"  snapshot_id: {snap.snapshot_id}")
print(f"  trigger:     {snap.trigger}")
print(f"  entities:    {snap.metadata.get('entities')}")
print(f"  triples:     {snap.metadata.get('triples')}")

# Take a second snapshot
snap2 = snap_mgr.take_snapshot(
    store, run_id=r2.run_id, trigger="post_run"
)
print(f"\n  Total snapshots: {len(snap_mgr.list_snapshots())}")

# Diff
diff = snap_mgr.diff_snapshots(snap.snapshot_id, snap2.snapshot_id)
print(f"  Entities added:   {len(diff.get('entities_added', []))}")
print(f"  Entities removed: {len(diff.get('entities_removed', []))}")
print("  (Same store, so no changes expected)")

In [ ]:
# Lab 20.1b -- RegressionSuite: promote failures
from knowlytix.harness.testing.replay import RegressionSuite, compare_runs

suite = RegressionSuite(suite_dir="monograph/lab_regression")
failed_ids = suite.promote_failures(r1, result)

print("=== RegressionSuite ===")
print(f"  Promoted failures: {len(failed_ids)}")
scenarios = suite.list_scenarios()
print(f"  Total scenarios:   {len(scenarios)}")
if scenarios:
    print(f"  Sample IDs: {scenarios[:3]}")

In [ ]:
# Lab 20.1c -- compare_runs
comparison = compare_runs(result, result, store=store)

print("=== Run Comparison (same result vs itself) ===")
print(f"  accuracy_delta:  {comparison.accuracy_delta:.4f}")
print(f"  regressions:     {len(comparison.regressions)}")
print(f"  improvements:    {len(comparison.improvements)}")
print(f"  stable_pass:     {comparison.stable_pass}")
print(f"  stable_fail:     {comparison.stable_fail}")
print(f"  regression_rate: {comparison.regression_rate:.4f}")
print(f"  semantic_drift:  {comparison.semantic_drift:.4f}")

In [ ]:
# Lab 20.1d -- ArtifactRegistry: version test artifacts
from knowlytix.harness.testing.versioning import ArtifactRegistry, VersionedArtifact

registry = ArtifactRegistry(
    artifacts_dir="monograph/lab_artifacts",
    artifact_type="fault_profile",
)

# Create and add an artifact
art = VersionedArtifact(
    artifact_type="fault_profile",
    content={"fault_rate": 0.3, "target_tools": ["data_lookup"]},
    author="ch20_notebook",
)
art_id = registry.add(art)

print("=== ArtifactRegistry ===")
print(f"  artifact_id:   {art_id}")
print(f"  version:       {art.version}")
print(f"  content_hash:  {art.content_hash[:20]}...")
print(f"  integrity:     {art.verify_integrity()}")

# Update artifact
art.content["fault_rate"] = 0.5
art.content["target_tools"].append("risk_check")
new_ver = registry.update(art)
print(f"\n  Updated to version {new_ver}")
print(f"  Versions available: {registry.list_versions(art_id)}")
print(f"  Total artifacts: {len(registry.list_all())}")

## Lab 20.2 — Performance and Cost Monitoring

GMSH encodes performance and cost metrics as phases on $S^1$,
enabling geometric SLA compliance measurement.

In [ ]:
# Lab 20.2a -- PerformanceProfiler
from knowlytix.harness.testing.profiling import PerformanceProfiler
import random

profiler = PerformanceProfiler(store=store, sla_latency_ms=5000.0)
random.seed(42)
tools = ["data_lookup", "risk_check", "create_ticket", "approve_action"]
for i in range(50):
    tool = tools[i % len(tools)]
    latency = max(100, random.gauss(2000, 800))
    profiler.record(tool_name=tool, latency_ms=latency)

perf_report = profiler.report()
print("=== PerformanceProfiler ===")
print(f"  Records:      {perf_report.n_records}")
print(f"  SLA target:   {perf_report.sla_latency_ms:.0f} ms")
print(f"  SLA pass:     {perf_report.sla_pass_rate:.1%}")
print(f"  Mean latency: {perf_report.mean_latency_ms:.1f} ms")
print(f"  P50:          {perf_report.p50_latency_ms:.1f} ms")
print(f"  P95:          {perf_report.p95_latency_ms:.1f} ms")
print(f"  P99:          {perf_report.p99_latency_ms:.1f} ms")
print(f"  Mean tension: {perf_report.mean_sla_tension:.4f}")

print("\n  Per-tool breakdown:")
for tool, stats in perf_report.per_tool.items():
    print(f"    {tool:<20s}: mean={stats['mean_latency_ms']:.0f}ms, "
          f"sla_pass={stats['sla_pass_rate']:.0%}")

In [ ]:
# Lab 20.2b -- CostTracker
from knowlytix.harness.testing.cost_tracker import CostTracker

tracker = CostTracker(store=store, budget_usd=5.0)
tracker.record("data_lookup", input_tokens=500, output_tokens=200)
tracker.record("risk_check", input_tokens=800, output_tokens=400)
tracker.record("create_ticket", input_tokens=300, output_tokens=100)
tracker.record("approve_action", input_tokens=1200, output_tokens=600)

cost_summary = tracker.summary()
print("=== CostTracker ===")
print(f"  Records:       {cost_summary.n_records}")
print(f"  Total tokens:  {cost_summary.total_tokens:,}")
print(f"  Total cost:    ${cost_summary.total_cost_usd:.4f}")
print(f"  Budget:        ${cost_summary.budget_usd:.2f}")
print(f"  Utilization:   {cost_summary.budget_utilization:.2%}")
print(f"  Within budget: {tracker.within_budget}")
print(f"  Budget tension: {cost_summary.budget_tension:.4f}")

print("\n  Per-tool breakdown:")
for tool, stats in cost_summary.per_tool.items():
    print(f"    {tool:<20s}: ${stats['cost_usd']:.4f} "
          f"({stats['input_tokens']}in/{stats['output_tokens']}out)")

In [ ]:
# Summary
print("=== Chapter 20 Summary ===")
print(f"  Runs managed:        {len(run_mgr.list_runs())}")
print(f"  Release gate:        {'PASS' if gate.passed else 'FAIL'}")
print(f"  Tiered gates:        {sum(1 for g in tiered_gates if g.passed)}/"
      f"{len(tiered_gates)} passed")
print(f"  Audit packets:       {len(packets)}")
print(f"  Document sections:   {len(doc_rec.sections)}")
print(f"  Provenance links:    {len(provenance_links)}")
print(f"  Store snapshots:     {len(snap_mgr.list_snapshots())}")
print(f"  Regression scenarios: {len(suite.list_scenarios())}")
print(f"  Versioned artifacts: {len(registry.list_all())}")
print(f"  Performance records: {perf_report.n_records}")
print(f"  Cost records:        {cost_summary.n_records}")
print(f"  Total cost:          ${cost_summary.total_cost_usd:.4f}")

## Exercises

1. **Exercise 20.1.** Create a custom `RiskTierProfile` with
   `accuracy_threshold=0.60`, `geo_confidence_threshold=0.30` and a
   failure budget of `{"hallucination_detected": 5}`. Evaluate it
   against the benchmark result. Does this relaxed profile pass?
   What accuracy would the mock agent need to pass the advisory tier?

2. **Exercise 20.2.** Register a second document in the
   `ProvenanceRegistry` (use any markdown file). Link 5 triples from
   the GMS store to sections in each document. Use `get_provenance()`
   to trace a triple back to its source. Create two snapshots before
   and after ingesting the second document and use `diff_snapshots()`
   to identify the new entities.

3. **Exercise 20.3.** Run the benchmark twice with different evaluators
   (e.g., one with `deliberate_errors=True` and one without). Use
   `compare_runs()` to build a `RunComparison`. How many regressions
   and improvements are detected? Promote the regressions to the
   `RegressionSuite` and verify the scenario count.

## Summary

This chapter demonstrated the **governance and operations** layer
of the GMSH framework:

- **RunManager** tracks benchmark runs through a validated state
  machine (CREATED -> PREPARING -> EXECUTING -> EVALUATING -> COMPLETED).
- **RiskTierProfile** defines three standard tiers (advisory,
  recommendation, action_taking) with escalating thresholds.
- **TieredReleaseGate** evaluates per-tier pass/fail with specific
  checks on accuracy, geometric confidence, tension and failure budgets.
- **ProvenanceRegistry** traces the lineage from source document
  through sections to triples and verdicts.
- **SnapshotManager** captures versioned store states for reproducibility.
- **RegressionSuite** promotes failures as regression scenarios.
- **ArtifactRegistry** versions test artifacts with content hashing.
- **PerformanceProfiler** and **CostTracker** encode latency and cost
  as $S^1$ phases for geometric SLA compliance.